# 03 - Multi-Modelo
> Claude + embeddings + classificadores em pipelines hibridos

**Modulo:** `08_arquitetura` | **Feito com:** [Claude](https://claude.ai) (Anthropic)

---


In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
HAIKU  = 'claude-haiku-4-5-20251001'
SONNET = 'claude-sonnet-4-5'

def ask(prompt, system=None, model=HAIKU, max_tokens=1024):
    kw = dict(model=model, max_tokens=max_tokens,
              messages=[{'role':'user','content':prompt}])
    if system: kw['system'] = system
    return client.messages.create(**kw).content[0].text

print('Pronto!')

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb, time

emb=SentenceTransformer('all-MiniLM-L6-v2')
col=chromadb.Client().get_or_create_collection('hybrid')

DOCS=['Plano Basic: R$29/mes, 5GB.','Plano Pro: R$79/mes, 50GB, suporte 24/7.',
      'Cancelar: Configuracoes > Conta > Cancelar.']
col.upsert(ids=[f'd{i}'for i in range(len(DOCS))],
           embeddings=emb.encode(DOCS).tolist(),documents=DOCS)

def hibrido(q):
    t0=time.time()
    intencao=ask(f'Intencao: "{q}" - produto, suporte ou cancelamento?',model=HAIKU).strip()
    res=col.query(query_embeddings=emb.encode([q]).tolist(),n_results=2)
    ctx='\n'.join(res['documents'][0])
    resp=ask(f'{ctx}\n\nPergunta: {q}',model=HAIKU)
    return intencao, resp, round(time.time()-t0,2)

for q in ['Qual o plano mais barato?','Como cancelo minha conta?']:
    intencao,resp,dt=hibrido(q)
    print(f'\n[{intencao}] {dt}s\n>>> {q}\n{resp[:120]}')

## Exercicios
> Complete os exercicios abaixo.


In [ ]:
# Seu codigo aqui


## Proximos passos
- Proximo notebook do modulo
- [docs.anthropic.com](https://docs.anthropic.com)
